In [10]:
from tensorflow.keras.models import load_model
import pickle

# Load trained multimodal model
model = load_model("final_multimodal_model.keras")

# Load tokenizer used during training
with open("text_tokenizer.pkl", "rb") as handle:
    tokenizer = pickle.load(handle)

print("✅ Model and tokenizer loaded successfully.")


✅ Model and tokenizer loaded successfully.


In [11]:
import numpy as np
from tensorflow.keras.preprocessing import image
from tensorflow.keras.preprocessing.sequence import pad_sequences

def preprocess_image(img_path):
    """Load an image, resize to 224x224, normalize pixel values."""
    img = image.load_img(img_path, target_size=(224, 224))
    img_array = image.img_to_array(img) / 255.0
    return np.expand_dims(img_array, axis=0)

def preprocess_text(text):
    """Convert text to token sequence and pad to fixed length."""
    sequence = tokenizer.texts_to_sequences([text])
    return pad_sequences(sequence, maxlen=100)  # match training length


In [6]:
import time
import numpy as np
from tensorflow.keras.models import load_model

# Load your trained model
model = load_model("final_multimodal_model.keras")

# Create dummy image input (224x224x3) and dummy text input (100 tokens)
dummy_image = np.random.rand(1, 224, 224, 3)
dummy_text = np.random.rand(1, 100)  # adjust if embedding layer is used

start_time = time.time()
_ = model.predict([dummy_image, dummy_text], verbose=0)
end_time = time.time()

print(f"Inference Time: {(end_time - start_time)*1000:.2f} ms")


Inference Time: 2061.95 ms


In [12]:
import pandas as pd

# Load dataset
test_data = pd.read_csv("cleaned_augmented_dataset.csv")

# Pick the first sample for timing
sample = test_data.iloc[0]

# Process inputs
image_input = preprocess_image(sample["Image"])
text_input = preprocess_text(sample["Historical Notes"])

print("🖼 Image shape:", image_input.shape)
print("📝 Text sequence shape:", text_input.shape)


🖼 Image shape: (1, 224, 224, 3)
📝 Text sequence shape: (1, 100)


In [13]:
import time

start_time = time.time()
_ = model.predict([image_input, text_input], verbose=0)
end_time = time.time()

inference_time_ms = (end_time - start_time) * 1000
print(f"⚡ Inference Time: {inference_time_ms:.2f} ms")


⚡ Inference Time: 1865.10 ms


In [ ]:
# --- Performance Evaluation ---

import time
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
from sklearn.metrics import classification_report, confusion_matrix

# 1. Load model and tokenizer
model = load_model("final_multimodal_model.keras")
with open("text_tokenizer.pkl", "rb") as f:
    tokenizer = pickle.load(f)

# 2. Load dataset
df = pd.read_csv("cleaned_augmented_dataset.csv")

# Assuming you already have your test split defined
# For demonstration, let's just take the last 20% as test set
test_split = int(len(df) * 0.8)
test_df = df.iloc[test_split:]

# 3. Preprocessing functions
from tensorflow.keras.preprocessing.sequence import pad_sequences

def preprocess_image(img_path):
    img = image.load_img(img_path, target_size=(224, 224))
    img_array = image.img_to_array(img) / 255.0
    return np.expand_dims(img_array, axis=0)

def preprocess_text(text):
    seq = tokenizer.texts_to_sequences([text])
    return pad_sequences(seq, maxlen=100, padding='post', truncating='post')

# 4. Generate predictions
y_true = []
y_pred = []

class_labels = sorted(df["Age"].unique())  #

for _, row in test_df.iterrows():
    img_input = preprocess_image(row["Image"])
    text_input = preprocess_text(row["Historical Notes"])

    prediction = model.predict([img_input, text_input], verbose=0)
    predicted_class = class_labels[np.argmax(prediction)]

    y_true.append(row["Age"])
    y_pred.append(predicted_class)

# 5. Inference time measurement (single sample)
sample_img = preprocess_image(test_df.iloc[0]["Image"])
sample_text = preprocess_text(test_df.iloc[0]["Historical Notes"])

start_time = time.time()
_ = model.predict([sample_img, sample_text], verbose=0)
end_time = time.time()

single_inference_time = (end_time - start_time) * 1000
print(f"Single sample inference time: {single_inference_time:.2f} ms")

# 6. Confusion matrix
cm = confusion_matrix(y_true, y_pred, labels=class_labels)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_labels,
            yticklabels=class_labels)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix - Multimodal Model")
plt.show()

# 7. Classification report
report = classification_report(y_true, y_pred, labels=class_labels, target_names=class_labels, output_dict=True)
report_df = pd.DataFrame(report).transpose()
print(report_df)

# Optional: Save the metrics table to CSV so you can include it in your thesis
report_df.to_csv("performance_metrics.csv")
